# [8.2] Attribution Patching and EAP - Solutions

Reference validation notebook for the section-local attribution patching and EAP implementation.

In [ ]:
import json
import sys
from pathlib import Path

chapter = "chapter8_automated_circuits"
section = "part2_attribution_patching_eap"
root_dir = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / chapter).exists())
exercises_dir = root_dir / chapter / "exercises"
section_dir = exercises_dir / section

if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))
if str(exercises_dir) not in sys.path:
    sys.path.append(str(exercises_dir))

import part2_attribution_patching_eap.tests as tests
from chapter8_automated_circuits.exercises.part2_attribution_patching_eap import solutions

In [ ]:
tests.test_attribution_patch_scores_sums_non_component_dims(
    solutions.attribution_patch_scores,
)
tests.test_integrated_gradient_patch_scores_average_path_gradients(
    solutions.integrated_gradient_patch_scores,
)
tests.test_edge_attribution_scores_forms_upstream_downstream_matrix(
    solutions.edge_attribution_scores,
)
tests.test_exact_vs_approx_reports_measure_correlation_and_topk_overlap(
    solutions.score_correlation_report,
    solutions.topk_overlap_report,
)
tests.test_runtime_and_false_negative_reports_enforce_accountability(
    solutions.runtime_improvement_report,
    solutions.false_negative_report,
)
tests.test_notebook_contract(solutions.run_smoke_test)

In [ ]:
contract = solutions.run_smoke_test(cpu=True)
assert contract["attribution_scores"] == [1.0, 3.0]
assert contract["integrated_gradients"] == [1.0, 3.0]
assert contract["edge_scores"] == [[3.0, 0.0], [0.0, 8.0]]
assert contract["correlation"]["passes_threshold"]
assert contract["topk_overlap"]["passes_threshold"]
assert contract["runtime"]["passes_speedup"]
assert contract["false_negative"]["documented"]
contract

In [ ]:
report = json.loads((section_dir / "verification_report.json").read_text())
gpu = report["metrics"]["gpu_test"]
assert gpu["preflight_passed"]
assert gpu["model_name"] == "gelu-1l"
assert gpu["hf_revision"] == "bddc0e332f0ae84279e6a6a45d91b314899e1603"
assert gpu["tokenizer_revision"] == "0f6671571a20be9756b9991d978047c03b75e749"
assert gpu["hook_name"] == "blocks.0.hook_resid_post"
assert gpu["exact_best_position"] == gpu["target_position"]
assert gpu["attribution_best_position"] == gpu["target_position"]
assert gpu["ig_best_position"] == gpu["target_position"]
assert gpu["exact_final_recovery"] == 1.0
assert gpu["attribution_final_recovery"] >= 0.9
assert gpu["ig_final_recovery"] >= 0.95
assert gpu["exact_attribution_top1_overlap"] == 1.0
assert gpu["exact_ig_top1_overlap"] == 1.0
assert gpu["eap_top_edge_upstream_position"] == gpu["target_position"]
assert gpu["eap_top_edge_downstream_position"] == gpu["target_position"]
assert gpu["nonfinal_gradient_norm_max"] <= 1e-7
assert gpu["within_vram_budget"]
{key: gpu[key] for key in [
    "model_name",
    "exact_final_recovery",
    "attribution_final_recovery",
    "ig_final_recovery",
    "exact_attribution_correlation",
    "exact_ig_correlation",
    "eap_top_edge_score_abs",
    "peak_vram_gb",
]}